# Validate Before You Run It

Level 1, Python Fundamentals in GenAI Context.

**Problems in this notebook**

- Problem 3: Validate JSON Tool-Call Arguments
- Problem 6: Prompt Template Variable Validation

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Paste the AI prompt given (or write your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference solution only after you have tried.

The ideas behind these problems are in `02_validate_before_running_THEORY.md`.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 3: Validate JSON Tool-Call Arguments

**The situation**

An AI agent can call tools such as `create_ticket` or `send_email`. The model writes the arguments as JSON, and models sometimes invent fields, use the wrong type, or leave things out. Nothing should touch a customer system until the arguments pass a check.

**Your task**

Write `validate_tool_args(args, required, allowed)`. Return a list of error strings. An empty list means the call is safe to run.

**Example**

`args={"customer_id": "c1", "admin": True}`, `required={"customer_id": str}`, `allowed={"customer_id", "priority"}` -> `["unexpected field: admin"]`

**Ask before coding**

- Should extra fields be rejected, or quietly dropped?
- Do you want every error at once, or should we stop at the first one?
- Are nested objects in scope, or only flat arguments?

**Plan to say aloud**

1. For each required field, report it if it is missing.
2. If it is present, report it if the type is wrong.
3. Treat `True` or `False` as wrong when a number is expected.
4. Report every field that is not allowed.
5. Return the list. Empty means valid.

### Prompt for your AI assistant

```text
Write a Python function `validate_tool_args(args: dict, required: dict[str, type], allowed: set[str]) -> list[str]`. Return error strings in this order: first, for each required field in order, either `missing required field: <name>` or `wrong type for <name>: expected <type>, got <type>`; then `unexpected field: <name>` for each field in args that is in neither allowed nor required, sorted by name. A bool must not pass as an int. Never raise. Show only the function.
```

**Check the AI's code for**

- Uses `isinstance(value, int)`, which accepts `True` because bool is a kind of int in Python.
- Stops at the first error instead of returning all of them.
- Ignores unexpected fields, so an invented `admin: true` slips through.
- Raises an exception on bad input instead of returning errors.

In [ ]:
def validate_tool_args(args: dict, required: dict[str, type], allowed: set[str]) -> list[str]:
    """Check model-generated tool arguments. Return a list of errors ([] means valid).

    Rules, in this order:
    - For each required field (in order): "missing required field: <name>" or
      "wrong type for <name>: expected <type>, got <type>".
    - A bool does not count as an int.
    - Then "unexpected field: <name>" for each field in neither allowed nor required, sorted.
    """
    raise NotImplementedError

In [ ]:
REQUIRED = {"customer_id": str, "limit": int}
ALLOWED = {"customer_id", "limit", "priority"}
cases = [
    (({"customer_id": "c1", "limit": 5}, REQUIRED, ALLOWED), []),
    (({"customer_id": "c1", "limit": 5, "priority": "high"}, REQUIRED, ALLOWED), []),
    (({"customer_id": "c1", "limit": 5, "admin": True}, REQUIRED, ALLOWED), ["unexpected field: admin"]),
    (({"limit": 5}, REQUIRED, ALLOWED), ["missing required field: customer_id"]),
    (({"customer_id": 42, "limit": True}, REQUIRED, ALLOWED),
     ["wrong type for customer_id: expected str, got int", "wrong type for limit: expected int, got bool"]),
    (({}, REQUIRED, set()), ["missing required field: customer_id", "missing required field: limit"]),
]
run_checks(validate_tool_args, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def validate_tool_args_reference(args: dict, required: dict[str, type], allowed: set[str]) -> list[str]:
    errors: list[str] = []
    for name, expected in required.items():
        if name not in args:
            errors.append(f"missing required field: {name}")
            continue
        value = args[name]
        bool_posing_as_number = isinstance(value, bool) and expected is not bool
        if bool_posing_as_number or not isinstance(value, expected):
            errors.append(f"wrong type for {name}: expected {expected.__name__}, got {type(value).__name__}")
    for name in sorted(set(args) - allowed - set(required)):
        errors.append(f"unexpected field: {name}")
    return errors


assert run_checks(validate_tool_args_reference, cases)

**Explain it in 30 seconds**

> "Model output is untrusted, so I check tool arguments before anything runs. I look for missing required fields, wrong types, and fields the tool does not accept, and I return every problem as a list. An empty list is the only green light. I also block `True` sneaking in as a number, because in Python a bool counts as an int."

**Edge cases to mention**

- An invented field such as `admin` or `role`.
- `True` passed where a number is expected.
- Empty arguments for a tool that has required fields.
- Nested objects, which this flat check does not look inside.

**If they ask "how would this work in production?"**

- Use a schema library such as Pydantic or JSON Schema for nested arguments, and version each tool's schema.
- Refuse to run the tool on any error, and send the errors back to the model so it can try again.
- Give each tool a risk level: write and admin tools get stricter checks or a human approval step.

## Problem 6: Prompt Template Variable Validation

**The situation**

Each customer can edit its own prompt template, with placeholders like `{customer_name}`. A typo or an unexpected placeholder such as `{password}` can crash the prompt at runtime or pull the wrong data into it. You want to catch this when the template is saved, not when a live request fails.

**Your task**

Write `validate_prompt_template(template, allowed)`. Find every `{placeholder}`, normalize the name (trim spaces, lowercase), and return errors for empty placeholders and for names not in `allowed`.

**Example**

`"Hello { customer_name }, case {CASE_ID}"` with `allowed={"customer_name", "case_id"}` -> `[]`. `"Hi {password}"` with `allowed={"name"}` -> `["unexpected variable: password"]`

**Ask before coding**

- Is the syntax always single braces, or also `{{ }}`?
- Should names match ignoring case and spaces?
- Should we also report allowed variables the template never uses?

**Plan to say aloud**

1. Find all text between `{` and `}` with a regex.
2. Trim and lowercase each name.
3. Add one error if any placeholder is empty.
4. Compare the names with the allowed set.
5. Return each unexpected name once, sorted.

### Prompt for your AI assistant

```text
Write a Python function `validate_prompt_template(template: str, allowed: set[str]) -> list[str]`. Find placeholders with `re.findall(r"\{([^{}]*)\}", template)`. Trim and lowercase each name. If any placeholder is empty, add "empty placeholder" once, first. Then add "unexpected variable: <name>" for each distinct name not in allowed, sorted. Do not call .format() on the template. Show only the function.
```

**Check the AI's code for**

- Reports the same unexpected name twice when it appears twice in the template.
- Compares raw names, so `{ Customer_Name }` fails against `customer_name`.
- Tests the template with `.format()`, which crashes on the first unknown key instead of listing them all.
- Skips an empty `{}` silently, even though it breaks rendering.

In [ ]:
import re


def validate_prompt_template(template: str, allowed: set[str]) -> list[str]:
    """Return errors for a prompt template ([] means valid).

    Rules:
    - Placeholders look like {name}. Trim spaces and lowercase the name.
    - If any placeholder is empty, add "empty placeholder" once, first.
    - Then "unexpected variable: <name>" for each distinct name not in allowed, sorted.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("Hello { customer_name }, case {CASE_ID}", {"customer_name", "case_id"}), []),
    (("Hi {password}", {"name"}), ["unexpected variable: password"]),
    (("Hi {name}, again {name}", {"name"}), []),  # duplicates are fine
    (("Hi {} and {}", {"name"}), ["empty placeholder"]),  # reported once
    (("{b} {a} {b}", set()), ["unexpected variable: a", "unexpected variable: b"]),  # sorted, no repeats
    (("No placeholders here.", {"name"}), []),
]
run_checks(validate_prompt_template, cases)

### Reference solution

Try first. Then compare.

In [ ]:
import re


def validate_prompt_template_reference(template: str, allowed: set[str]) -> list[str]:
    names = [m.strip().lower() for m in re.findall(r"\{([^{}]*)\}", template)]
    errors: list[str] = []
    if "" in names:
        errors.append("empty placeholder")
    for name in sorted(set(names) - allowed - {""}):
        errors.append(f"unexpected variable: {name}")
    return errors


assert run_checks(validate_prompt_template_reference, cases)

**Explain it in 30 seconds**

> "Customers edit these templates, so I check them when they are saved. I pull out every name between braces, trim and lowercase it, and compare it with the variables we actually provide. Anything else, like `{password}`, is reported, and so is an empty `{}`. Catching it at save time means a live customer request never fails on a typo."

**Edge cases to mention**

- The same placeholder used twice: reported once.
- Case and spacing differences, like `{ CASE_ID }`.
- Empty braces `{}`.
- Unbalanced braces such as `{name`, which the regex will not match.

**If they ask "how would this work in production?"**

- Run the check on save and again on deploy, and block the deploy if it fails.
- Version every template so a bad edit can be rolled back quickly.
- Keep secrets out of the allowed list entirely, so no template can ever ask for them.

## Wrap-up

- Anything written by a model or a customer is only a proposal until it passes a check.
- Use an allow list: name what is permitted and reject everything else.
- Return all the errors at once, so the caller (or the model) can fix them in one go.